## Visualization of results for leave batch out classification

In [ ]:
import sys
sys.path.append('../')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
plt.rcParams['pdf.fonttype'] = 42
from utils.mypalette import day_palette, pairing_strategy_palette


order = ['hand-crafted', 'Pretrained\nResNet18', 'Standard SimCLR', 'uniDINO', 'CellPaint SSL', 'SI SimCLR']

# Load the results
# Change 'logreg_results.csv' to 'mlp_results.csv' as needed
res_dir = 'results/comparison_features/LOBO'
df = pd.read_csv(os.path.join(res_dir, 'logreg_results_debug.csv'))

# Combine task and model for plotting if needed
print(f"Loaded {len(df)} predictions.")
print(f"Experiments: {df['exp_code'].unique()}")

In [ ]:
# check how many rows for each mutation, condition combination
value_counts = df.groupby(['true']).size()
print(value_counts)

In [ ]:
value_counts = df.groupby(['exp_code']).size()
print(value_counts)

In [ ]:
value_counts = df.groupby(['classifier', 'test_batch']).size()
print(value_counts)

In [ ]:
# keep only rows with nan values
nan_df = df[df.isna().any(axis=1)]

In [ ]:
nan_df.exp_code.unique()

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

def calculate_aucs(data, group_cols=['exp_code', 'classifier', 'test_batch']):
    auc_results = []
    
    # Ensure group_cols is a list to prevent issues if a single string is passed
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    
    for keys, group in data.groupby(group_cols):
        # We need at least two classes to calculate AUC
        if group['true'].nunique() < 2:
            continue
            
        # 1. GENERALIZATION: Create a dictionary of the current group's identifiers
        # Pandas groupby returns a tuple for multiple cols, or a single value for one col
        # We ensure it's always a tuple for zipping
        current_keys = keys if isinstance(keys, tuple) else (keys,)
        row_data = dict(zip(group_cols, current_keys))
        
        try:
            # 2. ROBUST LOGIC: Retrieve the task/classifier by name, not index.
            # We assume 'classifier' is one of the group columns. 
            # If not, we fall back to looking at the first row of the group.
            if 'classifier' in row_data:
                task = row_data['classifier']
            else:
                task = group['classifier'].iloc[0]

            # Determine positive class based on task
            if task == 'ALS':
                pos_label = 'als'
            elif task == 'TIC high':
                pos_label = 'A1 high'
            else:
                pos_label = 'A1 low'
            
            # Calculate AUC
            current_auc = roc_auc_score(
                (group['true'] == pos_label).astype(int), 
                group['pred_prob']
            )

            # Add the calculated AUC to our row data
            row_data['auc'] = current_auc
            auc_results.append(row_data)

        except Exception as e:
            print(f"Error calculating AUC for group {row_data}: {e}")
            
    return pd.DataFrame(auc_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

auc_df = calculate_aucs(df)

# Create a FacetGrid: Separates Classifiers into columns
g = sns.catplot(
    data=auc_df,
    x='exp_code',
    y='auc',
    col='classifier',
    # set col order
    col_order=['ALS', 'TIC low', 'TIC high'],
    kind='box',          
    hue='exp_code',      
    palette=pairing_strategy_palette,
    dodge=False,
    height=6, 
    aspect=0.8,
    fliersize=0,         
    hue_order=order,
    order=order,
    sharey=True    
)

# Map the Stripplot on top of the Boxplot
g.map_dataframe(
    sns.stripplot,
    x='exp_code',
    y='auc',
    hue='test_batch',         # 3. Colors the DOTS by Batch
    palette=day_palette,
    dodge=False,         # Centers the dots (aligning with the centered boxes)
    order=order,
    hue_order=None,      # Let seaborn infer batch order, or specify if needed
    alpha=1,
    size=8,
    edgecolor='black',
    linewidth=0.5,
    marker='o'
)

# Customizing the layout
g.set_titles("{col_name}")  # Titles for each subplot
g.set_axis_labels("Embedding Experiment", "AUC")


#plt.ylim(0.5, 1.05)
for ax in g.axes.flat:
    # Rotate x-labels for readability
    for label in ax.get_xticklabels():
        label.set_rotation(45)

plt.tight_layout()
#plt.savefig(os.path.join(res_dir, 'auc_lobo.pdf'), dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for i, cl in enumerate(['ALS', 'TIC low', 'TIC high']):

    sns.barplot(x='test_batch', y='auc', data=auc_df[auc_df['classifier']==cl],
                palette=pairing_strategy_palette, ax=ax[i],
                hue='exp_code',
                hue_order=order
                )

    ax[i].set_ylabel('AUC')
    ax[i].set_xticklabels(ax[i].get_xticklabels())
    ax[i].axhline(0.5, color='black', linestyle='--')
    #ax[i].set_ylim([0.3, 1.01])
    ax[i].set_title(f'{cl}', y=1.05)
    ax[i].legend(title='Test batch')
    ax[i].spines['top'].set_visible(False)
    ax[i].spines['right'].set_visible(False)
    ax[i].legend_.remove()
plt.suptitle('AUCs for leave-batch-out-experiments')
plt.tight_layout()
#plt.savefig(f'all_leavebatchout.pdf')
plt.show()